# Equipo 1 — Potenciales de Nernst y GHK
**Proyecto GluA2-AMPA · Electrofisiología Molecular I · UdeG CUCEI 2026**

Este notebook calcula:
- Potenciales de equilibrio (Nernst) para Na⁺, K⁺ y Ca²⁺
- Potencial de inversión (V_rev) del receptor AMPA-GluA2 en modos R y Q
- Curva corriente-voltaje (I-V) comparativa

In [ ]:
import sys
sys.path.append('../shared')
import numpy as np
import matplotlib.pyplot as plt
from parametros_compartidos import *

print('Parámetros cargados correctamente')
print(f'Temperatura: {TEMP_CELSIUS} °C  ({TEMP_KELVIN:.2f} K)')

## 1. Potenciales de equilibrio (Nernst)

In [ ]:
def nernst(z, conc_o, conc_i, T_celsius=37.0):
    """Potencial de Nernst en mV.
    E = (61.5 / z) * log10(conc_o / conc_i)  a 37°C
    """
    factor = (R * (T_celsius + 273.15) / F) * np.log(10) * 1000  # mV
    return (factor / z) * np.log10(conc_o / conc_i)

E_Na = nernst(z=1,  conc_o=NA_O, conc_i=NA_I)
E_K  = nernst(z=1,  conc_o=K_O,  conc_i=K_I)
E_Ca = nernst(z=2,  conc_o=CA_O, conc_i=CA_I)

print(f'E_Na = {E_Na:.2f} mV  (esperado ≈ +67 mV)')
print(f'E_K  = {E_K:.2f} mV  (esperado ≈ −95 mV)')
print(f'E_Ca = {E_Ca:.2f} mV  (esperado ≈ +130 mV)')

## 2. Potencial de inversión GHK

In [ ]:
def ghk_vrev(P_Ca_P_Na, T_celsius=37.0):
    """Potencial de inversión del receptor AMPA-GluA2 (mV).
    Asume P_K = P_Na = 1 (relativo). P_Ca varía entre modos.
    """
    RT_F_mV = (R * (T_celsius + 273.15) / F) * 1000  # mV

    P_Na = P_NA_P_NA
    P_K  = P_K_P_NA
    P_Ca = P_Ca_P_Na  # relativo a P_Na

    num = P_Na * NA_O + P_K * K_O + 4 * P_Ca * CA_O
    den = P_Na * NA_I + P_K * K_I + 4 * P_Ca * CA_I

    return RT_F_mV * np.log(num / den)

V_rev_R = ghk_vrev(P_CA_P_NA_R)
V_rev_Q = ghk_vrev(P_CA_P_NA_Q)
delta_V = V_rev_Q - V_rev_R

print(f'V_rev Modo R (P_Ca/P_Na = {P_CA_P_NA_R}) = {V_rev_R:.2f} mV')
print(f'V_rev Modo Q (P_Ca/P_Na = {P_CA_P_NA_Q}) = {V_rev_Q:.2f} mV')
print(f'ΔV_rev (Q − R) = {delta_V:.2f} mV')
print()
print('>>> COMPARTIR ESTOS VALORES CON EQUIPOS 2 Y 3 <<<')

## 3. Curva I-V

In [ ]:
# Conductancia total (ejemplo: 15 sinapsis × 10 pS)
g_total_nS = 15 * G_UNIT_PS * 1e-3   # nS

Vm = np.arange(-100, 65, 5)  # mV

# I = g * (Vm - V_rev)
I_R = g_total_nS * (Vm - V_rev_R)   # pA (g en nS, V en mV → I en pA)
I_Q = g_total_nS * (Vm - V_rev_Q)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(Vm, I_R, 'b-o', markersize=4, label=f'Modo R  (V_rev = {V_rev_R:.1f} mV)')
ax.plot(Vm, I_Q, 'r-s', markersize=4, label=f'Modo Q  (V_rev = {V_rev_Q:.1f} mV)')
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.axvline(V_rev_R, color='b', linewidth=0.8, linestyle=':')
ax.axvline(V_rev_Q, color='r', linewidth=0.8, linestyle=':')
ax.set_xlabel('V_m (mV)', fontsize=12)
ax.set_ylabel('Corriente (pA)', fontsize=12)
ax.set_title('Curva I-V del receptor AMPA-GluA2\nModo R vs Modo Q', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figura2_curva_IV.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada: figura2_curva_IV.png')

## 4. Tabla resumen (Tabla 1 del manuscrito)

In [ ]:
import pandas as pd

tabla = pd.DataFrame({
    'Parámetro': ['E_Na (mV)', 'E_K (mV)', 'E_Ca (mV)', 'V_rev Modo R (mV)', 'V_rev Modo Q (mV)', 'ΔV_rev (mV)'],
    'Valor': [round(E_Na,2), round(E_K,2), round(E_Ca,2),
              round(V_rev_R,2), round(V_rev_Q,2), round(delta_V,2)]
})
print(tabla.to_string(index=False))
tabla.to_csv('tabla1_potenciales.csv', index=False)
print('\nTabla guardada: tabla1_potenciales.csv')